# 🧠 CHRONOS — ICU Risk Prediction Pipeline

**Sections covered:**
- D · Patient-level Splitting
- E · Leakage-safe Preprocessing
- F · Risk Modeling Layer
- G · Validation Strategy
- H · Clinical Utility Optimization
- I · Novel Method Layer (Continuous Risk Trajectory)
- J · Explainability (Temporal SHAP → Rank Attribution → Driver Stability)

**Data source:** `mimic-research-490610.icu_pipeline.icu_temporal_multievent_1h` on BigQuery

---
## 0 · Setup & BigQuery Data Load

In [ ]:
# Install required packages (run once)
# !pip install google-cloud-bigquery db-dtypes pandas numpy scikit-learn
#              xgboost lifelines shap matplotlib seaborn statsmodels

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── BigQuery connection ───────────────────────────────────────────────────────
from google.cloud import bigquery

PROJECT   = 'mimic-research-490610'
DATASET   = 'icu_pipeline'
TABLE     = 'icu_temporal_multievent_1h'
FULL_TABLE = f'{PROJECT}.{DATASET}.{TABLE}'

client = bigquery.Client(project=PROJECT)

query = f'SELECT * FROM `{FULL_TABLE}`'
df = client.query(query).to_dataframe()

print(f'Loaded {len(df):,} rows  ×  {df.shape[1]} columns')
df.head(3)

---
## D · Patient-level Splitting

Uses the **`patient_split_bucket`** column (0-99 hash) created by BigQuery.
- Train : buckets 0-69  (70 %)
- Val   : buckets 70-84 (15 %)
- Test  : buckets 85-99 (15 %)

All rows of a patient go to exactly one split → **no data leakage across patients**.

In [ ]:
# ── D1 · Patient-level split using pre-computed bucket ────────────────────────
train_df = df[df['patient_split_bucket'] < 70].copy()
val_df   = df[(df['patient_split_bucket'] >= 70) & (df['patient_split_bucket'] < 85)].copy()
test_df  = df[df['patient_split_bucket'] >= 85].copy()

for name, subset in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_pat = subset['subject_id'].nunique()
    n_row = len(subset)
    pos   = subset['label_12h'].mean()
    print(f'{name:6s} | patients={n_pat:5,} | rows={n_row:7,} | label_12h positive rate={pos:.3f}')

# Verify no patient overlap
assert not set(train_df.subject_id) & set(val_df.subject_id), 'LEAKAGE: train/val overlap'
assert not set(train_df.subject_id) & set(test_df.subject_id), 'LEAKAGE: train/test overlap'
assert not set(val_df.subject_id)   & set(test_df.subject_id), 'LEAKAGE: val/test overlap'
print('\n✅ No patient overlap across splits.')

In [ ]:
# ── D2 · Split distribution visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Patient-level Split Distributions', fontsize=14, fontweight='bold')

split_info = {'Train': train_df, 'Val': val_df, 'Test': test_df}
colors     = {'Train': '#4C72B0', 'Val': '#DD8452', 'Test': '#55A868'}

for ax, (name, sdf) in zip(axes, split_info.items()):
    event_counts = sdf.groupby('stay_id')['label_12h'].max()
    ax.hist(sdf.groupby('subject_id').size(), bins=30, color=colors[name], alpha=0.8, edgecolor='white')
    ax.set_title(f'{name} — rows per patient')
    ax.set_xlabel('Hours per patient'); ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: split_distribution.png')

---
## E · Leakage-safe Preprocessing

**All statistics (medians, means, std) are computed ONLY on the training set** and then applied to val/test.

In [ ]:
# ── E0 · Define feature columns ───────────────────────────────────────────────
ID_COLS   = ['subject_id','hadm_id','stay_id','patient_split_bucket',
             'prediction_time','intime','outtime']
META_COLS = ['race','gender','first_careunit','admission_type',
             'ventilation_status','hospital_expire_flag']
LABEL_COLS= ['label_2h','label_6h','label_12h',
             'event_type_2h','event_type_6h','event_type_12h',
             'event_time_2h','event_time_6h','event_time_12h',
             'observation_window_hours']

# Numeric predictors
FEATURE_COLS = [
    'hr','map_mean','sbp','dbp','rr','spo2','temp_c','pulse_pressure',
    'gcs_total','fio2_frac','pao2','paco2_abg','ph_abg','pf_ratio','sf_ratio',
    'vent_active','vasopressor_active','ne_equivalent_dose','crrt_active',
    'lactate','creatinine','platelets','bilirubin_total','wbc',
    'sodium','potassium','bicarbonate','hemoglobin','glucose',
    'urine_ml','urine_ml_per_kg','body_weight_kg',
    'sofa_resp','sofa_coag','sofa_liver','sofa_renal','sofa_cardio','sofa_cns','sofa_approx',
    'shock_index','delta_sofa_6h','ards_flag','aki_stage','aki_stage_creat','aki_stage_uo',
    'hours_since_admission','hours_since_infection',
    'charlson_comorbidity_index','oasis','oasis_prob','sapsii','sapsii_prob',
    'hr_mean_12h','hr_std_12h','map_min_12h','map_mean_12h',
    'lactate_max_12h','creatinine_max_12h','platelets_min_12h','bilirubin_max_12h',
    'urine_sum_12h','urine_per_kg_sum_12h','vasopressor_any_12h','ne_dose_mean_12h',
    'vent_any_12h','sofa_max_12h','delta_sofa_mean_12h','aki_stage_max_12h',
    'pf_ratio_min_12h','sf_ratio_min_12h','shock_index_max_12h','observed_hours_in_window'
]

# Keep only columns that actually exist in the dataframe
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'Using {len(FEATURE_COLS)} numeric feature columns.')

In [ ]:
# ── E1 · Train-only Median Imputation ─────────────────────────────────────────
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
imputer.fit(train_df[FEATURE_COLS])

X_train = imputer.transform(train_df[FEATURE_COLS])
X_val   = imputer.transform(val_df[FEATURE_COLS])
X_test  = imputer.transform(test_df[FEATURE_COLS])

# Missingness indicator matrix (before imputation)
miss_train = train_df[FEATURE_COLS].isnull().astype(int).values
miss_val   = val_df[FEATURE_COLS].isnull().astype(int).values
miss_test  = test_df[FEATURE_COLS].isnull().astype(int).values

print('Imputation done.')
print(f'  X_train shape : {X_train.shape}')
print(f'  X_val   shape : {X_val.shape}')
print(f'  X_test  shape : {X_test.shape}')

In [ ]:
# ── E2 · Train-only Standardisation ──────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train_sc = scaler.transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Standardisation done (mean=0, std=1 computed from train only).')

In [ ]:
# ── E3 · Feature Engineering ──────────────────────────────────────────────────
# Add missingness indicators as extra features (E4 — Missingness Modelling)
HIGH_MISS_THRESHOLD = 0.20  # columns with >20% missing in train

miss_rates = train_df[FEATURE_COLS].isnull().mean()
HIGH_MISS_COLS = miss_rates[miss_rates > HIGH_MISS_THRESHOLD].index.tolist()
hi_idx = [FEATURE_COLS.index(c) for c in HIGH_MISS_COLS]

import numpy as np

def add_missingness(X, miss_matrix, hi_idx):
    """Append binary missingness indicators for high-missingness columns."""
    return np.hstack([X, miss_matrix[:, hi_idx]])

X_train_fe = add_missingness(X_train_sc, miss_train, hi_idx)
X_val_fe   = add_missingness(X_val_sc,   miss_val,   hi_idx)
X_test_fe  = add_missingness(X_test_sc,  miss_test,  hi_idx)

FEATURE_NAMES_FE = FEATURE_COLS + [f'miss_{c}' for c in HIGH_MISS_COLS]

print(f'High-missingness columns (>{HIGH_MISS_THRESHOLD*100:.0f}%): {HIGH_MISS_COLS}')
print(f'Augmented feature dim: {X_train_fe.shape[1]}')

# Labels
y_train_12 = train_df['label_12h'].values
y_val_12   = val_df['label_12h'].values
y_test_12  = test_df['label_12h'].values

y_train_6  = train_df['label_6h'].values
y_val_6    = val_df['label_6h'].values
y_test_6   = test_df['label_6h'].values

---
## F · Risk Modeling Layer

Three complementary models trained for the **12-hour prediction horizon**:
1. **Logistic Regression** — linear baseline
2. **XGBoost** — gradient-boosted non-linear model
3. **CoxPH survival model** — time-to-event via lifelines

In [ ]:
# ── F1 · Logistic Regression ─────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

lr = LogisticRegression(max_iter=1000, C=0.1, class_weight='balanced', random_state=42)
lr.fit(X_train_fe, y_train_12)

lr_proba_val  = lr.predict_proba(X_val_fe)[:,1]
lr_proba_test = lr.predict_proba(X_test_fe)[:,1]

print('Logistic Regression (12h horizon)')
print(f'  Val  AUROC={roc_auc_score(y_val_12,  lr_proba_val):.4f}  AUPRC={average_precision_score(y_val_12,  lr_proba_val):.4f}')
print(f'  Test AUROC={roc_auc_score(y_test_12, lr_proba_test):.4f}  AUPRC={average_precision_score(y_test_12, lr_proba_test):.4f}')

In [ ]:
# ── F2 · XGBoost ──────────────────────────────────────────────────────────────
import xgboost as xgb

# Class imbalance weight
neg = (y_train_12 == 0).sum()
pos = (y_train_12 == 1).sum()
scale_pos = neg / pos

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    scale_pos_weight=scale_pos,
    eval_metric='aucpr',
    early_stopping_rounds=30,
    random_state=42,
    use_label_encoder=False
)

xgb_model.fit(
    X_train_fe, y_train_12,
    eval_set=[(X_val_fe, y_val_12)],
    verbose=50
)

xgb_proba_val  = xgb_model.predict_proba(X_val_fe)[:,1]
xgb_proba_test = xgb_model.predict_proba(X_test_fe)[:,1]

print('\nXGBoost (12h horizon)')
print(f'  Val  AUROC={roc_auc_score(y_val_12,  xgb_proba_val):.4f}  AUPRC={average_precision_score(y_val_12,  xgb_proba_val):.4f}')
print(f'  Test AUROC={roc_auc_score(y_test_12, xgb_proba_test):.4f}  AUPRC={average_precision_score(y_test_12, xgb_proba_test):.4f}')

In [ ]:
# ── F3 · Cox Proportional Hazards Survival Model ─────────────────────────────
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

# Build a patient-level survival frame from training set
# Duration = hours_since_admission at last observed row; event = hospital_expire_flag
surv_train = (
    train_df
    .groupby('stay_id')
    .agg(
        duration   = ('hours_since_admission', 'max'),
        event      = ('hospital_expire_flag',  'max'),
        sofa_max   = ('sofa_approx',           'max'),
        charlson   = ('charlson_comorbidity_index', 'first'),
        oasis      = ('oasis',                  'first'),
        sapsii     = ('sapsii',                 'first'),
        age        = ('anchor_age',             'first'),
        vasop_any  = ('vasopressor_any_12h',    'max'),
        vent_any   = ('vent_any_12h',           'max'),
        lactate_max= ('lactate_max_12h',        'max'),
    )
    .reset_index()
    .dropna()
)

COX_FEATS = ['duration','event','sofa_max','charlson','oasis','sapsii',
             'age','vasop_any','vent_any','lactate_max']

cph = CoxPHFitter(penalizer=0.1)
cph.fit(surv_train[COX_FEATS], duration_col='duration', event_col='event')
cph.print_summary()

# C-index on test patients
def build_surv_df(split_df):
    return (
        split_df.groupby('stay_id')
        .agg(duration=('hours_since_admission','max'),
             event=('hospital_expire_flag','max'),
             sofa_max=('sofa_approx','max'),
             charlson=('charlson_comorbidity_index','first'),
             oasis=('oasis','first'), sapsii=('sapsii','first'),
             age=('anchor_age','first'),
             vasop_any=('vasopressor_any_12h','max'),
             vent_any=('vent_any_12h','max'),
             lactate_max=('lactate_max_12h','max'))
        .reset_index().dropna()
    )

surv_test = build_surv_df(test_df)
risk_scores = cph.predict_partial_hazard(surv_test[COX_FEATS])
c_idx = concordance_index(surv_test['duration'], -risk_scores, surv_test['event'])
print(f'\nCoxPH Test C-index: {c_idx:.4f}')

---
## G · Validation Strategy

- **G1** Nested patient-level cross-validation on train set
- **G2** Temporal performance metrics (AUROC, AUPRC, Brier, calibration)
- **G3** Lead-time analysis & alarm burden

In [ ]:
# ── G1 · Nested Patient-level Cross Validation ────────────────────────────────
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

N_OUTER = 5
gkf = GroupKFold(n_splits=N_OUTER)
groups = train_df['subject_id'].values

cv_aurocs = []
for fold, (tr_idx, vl_idx) in enumerate(gkf.split(X_train_fe, y_train_12, groups)):
    fold_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.08,
        scale_pos_weight=scale_pos, random_state=42, use_label_encoder=False, eval_metric='logloss'
    )
    fold_model.fit(X_train_fe[tr_idx], y_train_12[tr_idx], verbose=False)
    prob = fold_model.predict_proba(X_train_fe[vl_idx])[:,1]
    auroc = roc_auc_score(y_train_12[vl_idx], prob)
    cv_aurocs.append(auroc)
    print(f'  Fold {fold+1}/{N_OUTER}  AUROC={auroc:.4f}')

print(f'\nNested CV  mean AUROC = {np.mean(cv_aurocs):.4f} ± {np.std(cv_aurocs):.4f}')

In [ ]:
# ── G2 · Temporal Performance Metrics ─────────────────────────────────────────
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              brier_score_loss, roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve

def full_metrics(y_true, y_prob, label=''):
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    print(f'{label:20s}  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  Brier={brier:.4f}')
    return {'auroc': auroc, 'auprc': auprc, 'brier': brier, 'y_true': y_true, 'y_prob': y_prob}

metrics = {}
print('=== Test-set performance ===')
metrics['LR_12h']  = full_metrics(y_test_12, lr_proba_test,  'LR  (12h)')
metrics['XGB_12h'] = full_metrics(y_test_12, xgb_proba_test, 'XGB (12h)')

# Plot ROC + PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for key, color in [('LR_12h','steelblue'),('XGB_12h','tomato')]:
    m = metrics[key]
    fpr, tpr, _ = roc_curve(m['y_true'], m['y_prob'])
    prec, rec, _ = precision_recall_curve(m['y_true'], m['y_prob'])
    ax1.plot(fpr, tpr, label=f"{key} (AUC={m['auroc']:.3f})", color=color)
    ax2.plot(rec, prec, label=f"{key} (AUPRC={m['auprc']:.3f})", color=color)

ax1.plot([0,1],[0,1],'k--'); ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR'); ax1.set_title('ROC Curve')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision'); ax2.set_title('Precision-Recall Curve')
ax1.legend(); ax2.legend()
plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── G3 · Lead-time Analysis & Alarm Burden ────────────────────────────────────
THRESHOLD = 0.35   # operational alarm threshold (tune via G task)

test_eval = test_df[['stay_id','prediction_time','hours_since_admission',
                      'label_12h','event_time_12h']].copy()
test_eval['xgb_prob']  = xgb_proba_test
test_eval['xgb_alarm'] = (xgb_proba_test >= THRESHOLD).astype(int)

# Lead time: for true positive hours, how many hours before event was first alarm?
tp_rows = test_eval[(test_eval['label_12h']==1) & (test_eval['xgb_alarm']==1)].copy()
tp_rows['event_time_12h'] = pd.to_datetime(tp_rows['event_time_12h'])
tp_rows['prediction_time'] = pd.to_datetime(tp_rows['prediction_time'])
tp_rows['lead_hours'] = (tp_rows['event_time_12h'] - tp_rows['prediction_time']).dt.total_seconds() / 3600

# First alarm per stay
first_alarm = (
    tp_rows[tp_rows['lead_hours'] > 0]
    .groupby('stay_id')['lead_hours']
    .max()   # max lead = earliest alarm
)

alarm_rate = test_eval.groupby('stay_id')['xgb_alarm'].mean()  # alarms per hour

print(f'Threshold = {THRESHOLD}')
print(f'Median lead-time (h) : {first_alarm.median():.1f}')
print(f'Mean alarm rate/hr   : {alarm_rate.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(first_alarm, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Lead-time Distribution (TP alarms)'); axes[0].set_xlabel('Hours before event')
axes[1].hist(alarm_rate, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Alarm Burden per Patient'); axes[1].set_xlabel('Fraction of hours with alarm')
plt.tight_layout()
plt.savefig('leadtime_alarm.png', dpi=150, bbox_inches='tight')
plt.show()

---
## H · Clinical Utility Optimization

- **H1** Decision Curve Analysis (DCA)
- **H2** Net Benefit
- **H3** Treatment Threshold sweep

In [ ]:
# ── H1-H3 · Decision Curve Analysis + Net Benefit ─────────────────────────────
def net_benefit(y_true, y_prob, threshold):
    """NB = TPR - (threshold / (1-threshold)) * FPR  (scaled by prevalence)"""
    n = len(y_true)
    pred = (y_prob >= threshold).astype(int)
    tp = ((pred == 1) & (y_true == 1)).sum()
    fp = ((pred == 1) & (y_true == 0)).sum()
    odds = threshold / (1 - threshold + 1e-9)
    return (tp - odds * fp) / n

thresholds = np.linspace(0.02, 0.60, 60)
prev = y_test_12.mean()

nb_xgb  = [net_benefit(y_test_12, xgb_proba_test, t) for t in thresholds]
nb_lr   = [net_benefit(y_test_12, lr_proba_test,  t) for t in thresholds]
nb_all  = [prev - (t/(1-t+1e-9))*(1-prev) for t in thresholds]  # treat-all baseline
nb_none = [0.0] * len(thresholds)  # treat-none

plt.figure(figsize=(10, 5))
plt.plot(thresholds, nb_xgb,  label='XGBoost',     color='tomato',    lw=2)
plt.plot(thresholds, nb_lr,   label='Log. Reg.',    color='steelblue', lw=2)
plt.plot(thresholds, nb_all,  label='Treat All',    color='gray',      lw=1.5, linestyle='--')
plt.plot(thresholds, nb_none, label='Treat None',   color='black',     lw=1.5, linestyle=':')
plt.axvline(x=THRESHOLD, color='orange', linestyle='-.', label=f'Operational threshold={THRESHOLD}')
plt.xlabel('Threshold probability'); plt.ylabel('Net Benefit')
plt.title('Decision Curve Analysis — 12h Horizon')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dca.png', dpi=150, bbox_inches='tight')
plt.show()

# Optimal threshold by max net benefit
best_thresh = thresholds[np.argmax(nb_xgb)]
print(f'Optimal threshold (max NB): {best_thresh:.3f}  |  NB = {max(nb_xgb):.4f}')

---
## I · Novel Method Layer

**Continuous Risk Trajectory** → rolling predicted-risk over a patient's ICU stay  
**Dynamic Patient Ranking** → hourly ICU-wide rank by predicted risk  
**Intervention Priority Score** → weighted composite ranking

In [ ]:
# ── I1 · Continuous Risk Trajectory ──────────────────────────────────────────
test_traj = test_df[['stay_id','subject_id','prediction_time','hours_since_admission',
                      'label_12h','hospital_expire_flag']].copy()
test_traj['risk_12h']  = xgb_proba_test
test_traj['prediction_time'] = pd.to_datetime(test_traj['prediction_time'])

# Smooth trajectory with exponential weighted mean (EWM) — clinical smoothing
test_traj = test_traj.sort_values(['stay_id','prediction_time'])
test_traj['risk_smooth'] = (
    test_traj.groupby('stay_id')['risk_12h']
    .transform(lambda s: s.ewm(span=4, adjust=False).mean())
)

# Plot trajectories for 6 random patients
sample_stays = test_traj['stay_id'].drop_duplicates().sample(6, random_state=1).values
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharey=True)
fig.suptitle('Continuous Risk Trajectories (XGBoost 12h)', fontsize=13, fontweight='bold')

for ax, sid in zip(axes.flat, sample_stays):
    pat = test_traj[test_traj.stay_id == sid].sort_values('hours_since_admission')
    died = pat['hospital_expire_flag'].iloc[0]
    ax.fill_between(pat['hours_since_admission'], pat['risk_smooth'], alpha=0.25, color='tomato')
    ax.plot(pat['hours_since_admission'], pat['risk_smooth'], color='tomato', lw=1.8)
    ax.axhline(THRESHOLD, color='gray', linestyle='--', lw=1, label='Threshold')
    ax.set_title(f'Stay {sid} | Died={bool(died)}', fontsize=9)
    ax.set_xlabel('Hours since admission'); ax.set_ylabel('P(event|12h)')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('risk_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── I2 · Dynamic Patient Ranking ──────────────────────────────────────────────
# Rank patients within each prediction hour (lower rank = higher risk)
test_traj['rank_in_hour'] = (
    test_traj.groupby('prediction_time')['risk_smooth']
    .rank(ascending=False, method='first')
)
test_traj['n_patients_in_hour'] = (
    test_traj.groupby('prediction_time')['stay_id'].transform('nunique')
)
test_traj['percentile_rank'] = (
    (test_traj['rank_in_hour'] / test_traj['n_patients_in_hour']) * 100
)

# Summary: among patients who had events, what was their median rank percentile?
event_patients = test_traj[test_traj['label_12h'] == 1]
no_event       = test_traj[test_traj['label_12h'] == 0]
print(f'Median rank percentile — event patients   : {event_patients["percentile_rank"].median():.1f}%')
print(f'Median rank percentile — no-event patients: {no_event["percentile_rank"].median():.1f}%')

In [ ]:
# ── I3 · Intervention Priority Score ─────────────────────────────────────────
# IPS = weighted combination of risk trajectory and acuity features
test_ips = test_df[['stay_id','prediction_time',
                     'sofa_approx','delta_sofa_6h','shock_index',
                     'vasopressor_active','vent_active']].copy()
test_ips['risk_12h'] = xgb_proba_test

# Normalise each component to [0,1] on test set (purely for ranking — no train leakage concern here)
def minmax(s):
    r = s.max() - s.min()
    return (s - s.min()) / (r + 1e-9)

test_ips['sofa_norm']    = minmax(test_ips['sofa_approx'].fillna(0))
test_ips['delta_norm']   = minmax(test_ips['delta_sofa_6h'].fillna(0).clip(lower=0))
test_ips['shock_norm']   = minmax(test_ips['shock_index'].fillna(0))
test_ips['risk_norm']    = minmax(test_ips['risk_12h'])

# Weighted IPS
W_RISK=0.50; W_SOFA=0.20; W_DELTA=0.20; W_SHOCK=0.10
test_ips['IPS'] = (
    W_RISK  * test_ips['risk_norm']  +
    W_SOFA  * test_ips['sofa_norm']  +
    W_DELTA * test_ips['delta_norm'] +
    W_SHOCK * test_ips['shock_norm']
)

print('Intervention Priority Score summary:')
print(test_ips['IPS'].describe().round(4))

plt.figure(figsize=(8,4))
plt.hist(test_ips['IPS'], bins=40, color='mediumpurple', edgecolor='white', alpha=0.85)
plt.title('Intervention Priority Score Distribution'); plt.xlabel('IPS'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig('ips_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## J · Explainability

- **J1** Temporal SHAP — per-hour SHAP values
- **J2** Rank Attribution — which features drive high-priority rankings
- **J3** Driver Stability — how consistent are top SHAP drivers across time windows

In [ ]:
# ── J1 · Temporal SHAP ────────────────────────────────────────────────────────
import shap

# Use a subsample of test set for SHAP (computationally feasible)
shap_sample_idx = np.random.default_rng(42).choice(len(X_test_fe), size=min(2000, len(X_test_fe)), replace=False)
X_shap = X_test_fe[shap_sample_idx]

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

shap_df = pd.DataFrame(shap_values, columns=FEATURE_NAMES_FE)
shap_df['stay_id']            = test_df['stay_id'].iloc[shap_sample_idx].values
shap_df['hours_since_admission'] = test_df['hours_since_admission'].iloc[shap_sample_idx].values

# Global SHAP beeswarm
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_NAMES_FE,
                  max_display=20, show=False)
plt.title('SHAP Global Feature Importance (XGBoost 12h)', fontsize=12)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

# SHAP bar chart (mean absolute)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({'feature': FEATURE_NAMES_FE, 'mean_abs_shap': mean_abs_shap})
importance_df = importance_df.sort_values('mean_abs_shap', ascending=False).head(20)
print('Top 20 features by mean |SHAP|:')
print(importance_df.to_string(index=False))

In [ ]:
# ── J1 (cont.) · Temporal SHAP — SHAP value evolution over ICU stay ───────────
TOP_K = 8
top_features = importance_df['feature'].head(TOP_K).tolist()

# Bin hours
shap_df['hour_bin'] = pd.cut(shap_df['hours_since_admission'], bins=[0,6,12,24,48,120], right=True,
                              labels=['0-6h','6-12h','12-24h','24-48h','48h+'])

# Mean SHAP per time bin per feature
temporal_shap = (
    shap_df.groupby('hour_bin')[top_features]
    .mean()
)

fig, ax = plt.subplots(figsize=(12, 5))
temporal_shap.T.plot(kind='bar', ax=ax, colormap='tab10', width=0.75, edgecolor='white')
ax.set_title('Temporal SHAP — Mean Feature Effect by ICU Hour Bin', fontsize=12)
ax.set_xlabel('Feature'); ax.set_ylabel('Mean SHAP value')
ax.legend(title='Hour bin', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('temporal_shap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── J2 · Rank Attribution ─────────────────────────────────────────────────────
# For high-priority rows (top 20% IPS), what SHAP features dominate?
shap_idx_in_test   = shap_sample_idx
ips_sample         = test_ips['IPS'].iloc[shap_idx_in_test].values
high_priority_mask = ips_sample >= np.quantile(ips_sample, 0.80)

hi_shap = shap_values[high_priority_mask]
lo_shap = shap_values[~high_priority_mask]

mean_hi = np.abs(hi_shap).mean(axis=0)
mean_lo = np.abs(lo_shap).mean(axis=0)

rank_attr = pd.DataFrame({
    'feature'          : FEATURE_NAMES_FE,
    'hi_priority_shap' : mean_hi,
    'lo_priority_shap' : mean_lo,
    'ratio'            : mean_hi / (mean_lo + 1e-9)
}).sort_values('hi_priority_shap', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(rank_attr))
w = 0.38
ax.barh(x + w/2, rank_attr['hi_priority_shap'], w, label='High-priority (top 20% IPS)', color='tomato')
ax.barh(x - w/2, rank_attr['lo_priority_shap'], w, label='Lower-priority',             color='steelblue', alpha=0.7)
ax.set_yticks(x); ax.set_yticklabels(rank_attr['feature'])
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP|'); ax.set_title('Rank Attribution — High vs Low Priority Patients')
ax.legend()
plt.tight_layout()
plt.savefig('rank_attribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Features with highest ratio (extra important for high-priority patients):')
print(rank_attr[['feature','ratio']].head(10).to_string(index=False))

In [ ]:
# ── J3 · Driver Stability ─────────────────────────────────────────────────────
# Does the top-K set of SHAP drivers stay stable across early / mid / late ICU windows?

TOP_K_STABLE = 10

# Assign each shap row to a time window
shap_df['window'] = pd.cut(
    shap_df['hours_since_admission'],
    bins=[0, 12, 36, 9999],
    labels=['Early (0-12h)', 'Mid (12-36h)', 'Late (36h+)']
)

feat_cols = FEATURE_NAMES_FE
windows   = ['Early (0-12h)', 'Mid (12-36h)', 'Late (36h+)']

window_top = {}
for w in windows:
    sub = shap_df[shap_df['window'] == w][feat_cols]
    if len(sub) < 5:
        window_top[w] = set()
        continue
    top_feats = (
        sub.abs().mean().sort_values(ascending=False)
        .head(TOP_K_STABLE).index.tolist()
    )
    window_top[w] = set(top_feats)

# Jaccard stability across windows
def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

print(f'Top-{TOP_K_STABLE} Driver Stability (Jaccard similarity):')
for i, w1 in enumerate(windows):
    for w2 in windows[i+1:]:
        j = jaccard(window_top[w1], window_top[w2])
        print(f'  {w1} vs {w2}  →  Jaccard = {j:.3f}')

print('\nTop drivers per window:')
for w in windows:
    print(f'  {w}: {sorted(window_top[w])}')

In [ ]:
# ── J3 (cont.) · Driver Stability Heatmap ────────────────────────────────────
# Show mean |SHAP| of top features across time windows as a heatmap
all_top = sorted(set().union(*window_top.values()))

heatmap_data = {}
for w in windows:
    sub = shap_df[shap_df['window'] == w][feat_cols]
    if len(sub) < 5:
        heatmap_data[w] = {f: 0.0 for f in all_top}
    else:
        means = sub[all_top].abs().mean()
        heatmap_data[w] = means.to_dict()

hmap = pd.DataFrame(heatmap_data, index=all_top)  # features × windows

fig, ax = plt.subplots(figsize=(9, max(4, len(all_top)*0.45)))
sns.heatmap(
    hmap, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5,
    ax=ax, cbar_kws={'label': 'Mean |SHAP|'}
)
ax.set_title(f'Driver Stability — Top-{TOP_K_STABLE} SHAP Features across ICU Time Windows', fontsize=11)
ax.set_xlabel('ICU Time Window'); ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('driver_stability_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Section J complete — Explainability pipeline done.')
print('   Outputs: shap_beeswarm.png, temporal_shap.png, rank_attribution.png, driver_stability_heatmap.png')

---
## ✅ Pipeline Complete — Summary

| Section | Status | Key Output |
|---------|--------|------------|
| D · Patient Split | ✅ | 70/15/15 split, no patient overlap |
| E · Preprocessing | ✅ | Median imputation + z-score (train-only), missingness indicators |
| F · Modeling      | ✅ | Logistic Regression, XGBoost, CoxPH |
| G · Validation    | ✅ | Nested 5-fold CV, AUROC/AUPRC/Brier, lead-time |
| H · Clinical Utility | ✅ | Decision Curve Analysis, Net Benefit, threshold sweep |
| I · Novel Method  | ✅ | Risk trajectories, dynamic ranking, IPS |
| J · Explainability | ✅ | Temporal SHAP, Rank Attribution, Driver Stability |

**Next:** Sections K (ICU Operational Simulation), L (Clinical Decision Support UI), M (Research Output)